# 03. Ансамбли, сравнение и интерпретация

Код восстановлен по PDF-отчёту. Результаты будут рассчитаны при выполнении; точное совпадение с отчётом не гарантируется. Запускайте ячейки сверху вниз.

In [ ]:
from pathlib import Path
import sys
ROOT = Path.cwd()
if not (ROOT / 'src' / 'project.py').exists():
    raise RuntimeError('Запустите Jupyter из корня проекта, где находятся src/ и README.md')
sys.path.insert(0, str(ROOT / 'src'))
from project import *
frame, (X_train, X_test, y_train, y_test) = load_split()

Полный запуск выполняет подбор по 5 фолдам: RF — 80/160 деревьев и глубина 8/None; GB — 60/120 деревьев и learning_rate 0.05/0.1. Это может занять длительное время. TUNE=False использует параметры из отчёта, но не воспроизводит процедуру подбора.

In [ ]:
TUNE = True
models = prepare_models(X_train, y_train, tune=TUNE)

Voting объединяет RF, GB и логистическую регрессию с равными весами (soft voting). Состав восстановлен по описанию: исходный код и веса Voting в PDF не приведены. Лучшая модель выбирается по CV до оценки test. CV после подбора на тех же фолдах может быть оптимистичной; это не вложенная CV.

In [ ]:
table, best_name = compare(models, X_train, X_test, y_train, y_test)
display(table.round(4))
print('Лучшая модель по CV:', best_name)

In [ ]:
for name in models:
    print('\n', name)
    print((ROOT/'results'/(name.replace(' ','_')+'.txt')).read_text(encoding='utf-8'))

Для сопоставления с отчётом важности и SHAP строим для Random Forest. Эти объяснения описывают поведение модели, а не причинное влияние. OOB после кодирования на всём train служит дополнительной оценкой.

In [ ]:
explain(models['Random Forest'], X_train)

После выполнения сравните results/metrics.csv с таблицей в README и обновите её фактически рассчитанными значениями. Порог равен 0.5; test не используется для настройки порога.